# Assignment Keras Intro
For this assignment, we will examine our keras MNIST network, and try to optimize it.   We will keras tuner to do this, following the strategy outlined in the book.



In [28]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook+pdf'
import time
t00 = time.time()

# Task 1: Get the data and import calc_performance_multi.
Get both the test and train data from Keras.

As we did in the in-class work, use the "Train" sample to form a train and validation dataset.   Keep the Test set separate and use it for final performance evaluation after training.

NOTE: User the smaller train set for all of the work in this notebook, to keep executiom times reasonable.

In [29]:
import tensorflow as tf
from tensorflow import keras
print(tf.__version__)
print(keras.__version__)
#
# Do these to reduce arning messages below
tf.get_logger().setLevel('ERROR')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# your code here
# your code here
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()
print("Train info all:",train_images.shape, train_labels.shape)
print("Test info all:",test_images.shape, test_labels.shape)

short = True
if short:
    train_images = train_images[:7000,:]
    train_labels = train_labels[:7000]

print()
print("Train info used:",train_images.shape, train_labels.shape)
print("Test info used:",test_images.shape, test_labels.shape)

train_images = train_images.astype('float32')/255
test_images = test_images.astype('float32')/255

train_labels_cat = keras.utils.to_categorical(train_labels)
test_labels_cat = keras.utils.to_categorical(test_labels)

from sklearn.model_selection import train_test_split
train_images_temp,val_images,train_labels_cat_temp,val_labels_cat = train_test_split(train_images,train_labels_cat, 
                                                                                         test_size=0.1, random_state=42)

2.19.1
3.10.0
Train info all: (60000, 28, 28) (60000,)
Test info all: (10000, 28, 28) (10000,)

Train info used: (7000, 28, 28) (7000,)
Test info used: (10000, 28, 28) (10000,)


In [30]:

from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
from sklearn import metrics
from sklearn.metrics import auc

from tabulate import tabulate

def calc_performance_multi(y_vals_true, y_vals_pred,labels):
#
# Get the numbers for the confusion matrix
# To get output: cf_matrix[true_label,pred_label]
    cf_matrix = confusion_matrix(y_vals_true, y_vals_pred, labels=labels)
#
# This is a graphic
    cf_disp = ConfusionMatrixDisplay(confusion_matrix=cf_matrix,display_labels=labels)
#
# Make the header row
    header = [""]
    for column_name in labels:
        header.append('Pred:' + str(column_name))
    table = [header]
#
# Now make the rows with the matrix
    for row_name in labels:
        row = ['True:'+str(row_name)]
        for column_name in labels:
            row_index = labels.index(row_name)
            column_index = labels.index(column_name)
            row.append(cf_matrix[row_index,column_index])
        table.append(row)
    
    print_table_type='fancy_grid'
    print_table = tabulate(table, headers='firstrow', tablefmt=print_table_type)
#
# Get the recall, precision, ands F1 for each individual label
# - return both the "string report" (which you can print)
# - and the "dictionary report" (which you can use for averages and so on)
    report = classification_report(y_vals_true,y_vals_pred,digits=4)
    report_dict = classification_report(y_vals_true,y_vals_pred,output_dict=True, digits=4)
#
    results = {"confusionMatrix":cf_matrix,
                    'confusion_matrix_display':cf_disp,
                    'confusion_matrix_print_table':print_table,      
                    "report":report,"report_dict":report_dict}
    return results

# Task 2: Modify "build_model" function 
Start with the version of build_model_tuner from the inclass module.

Make the following modifcations:
- Use 3 hidden layers instead of 1
- Add appropriate code to tune the number of neurons in each of these 3 layers.   The example in class had this:
   ```
  hp_n_neurons_h1 = hp.Int('n_neurons_h1', min_value=32, max_value=256, step=32)
  ```
    You will need to add 2 more with appropriate modifications.
  
- Add an additional value of 1e-1 to the list for hp_learning_rate

Ask me if you have questions!!

In [31]:
def build_model_tuner(hp):
    """
    Builds a Keras model with hyperparameters for Keras Tuner.
    """
    hp_n_neurons_h1 = hp.Int('n_neurons_h1', min_value=32, max_value=256, step=32)
    hp_n_neurons_h2 = hp.Int('n_neurons_h2', min_value=32, max_value=256, step=32)
    hp_n_neurons_h3 = hp.Int('n_neurons_h3', min_value=32, max_value=256, step=32)

    hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
        
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-1, 1e-2, 1e-3, 1e-4])
    
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd'])

    if hp_optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    elif hp_optimizer == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=hp_learning_rate)
  
    input_shape=[28,28]
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=input_shape))
    model.add(keras.layers.Flatten())

    model.add(keras.layers.Dense(units=hp_n_neurons_h1, activation=hp_activation))
    model.add(keras.layers.Dense(units=hp_n_neurons_h2, activation=hp_activation))
    model.add(keras.layers.Dense(units=hp_n_neurons_h3, activation=hp_activation))

    model.add(keras.layers.Dense(10, activation='softmax'))     
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# Task 3: Perform Keras Tuner Steps 2 and 3

Do Steps 2 and 3 just like in class:
- Run the tuner seacrh
- print out the summary and the best hyperparameters

In [32]:
import keras_tuner as kt
# We pass the model-building function directly to the tuner
t0 = time.time()
tuner = kt.RandomSearch(build_model_tuner,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=2,
    directory='keras_tuner_dir',
    project_name='mnist_tuning_assignment',
    overwrite=True
)
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)]
print("\nStarting Keras Tuner search...")
tuner.search(train_images_temp, train_labels_cat_temp, 
             epochs=10,
             batch_size=128,
             verbose=1,
             callbacks=callbacks,
             validation_data=(val_images, val_labels_cat))

print("Tuner search time:", time.time()-t0)

Trial 10 Complete [00h 00m 07s]
val_loss: 0.2549217715859413

Best val_loss So Far: 0.216463603079319
Total elapsed time: 00h 01m 05s
Tuner search time: 64.87144327163696


In [33]:
# Summarize and Display the Results ---

# your code here
print("\n--- Keras Tuner Search Complete ---")
tuner.results_summary()

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n--- Best Hyperparameters Found ---")
print(f"""
Number of Neurons Layer 1: {best_hps.get('n_neurons_h1')}
Number of Neurons Layer 2: {best_hps.get('n_neurons_h2')}
Number of Neurons Layer 3: {best_hps.get('n_neurons_h3')}
Activation Function: {best_hps.get('activation')}
Optimizer: {best_hps.get('optimizer')}
Learning Rate: {best_hps.get('learning_rate')}
""")



--- Keras Tuner Search Complete ---
Results summary
Results in keras_tuner_dir/mnist_tuning_assignment
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 03 summary
Hyperparameters:
n_neurons_h1: 192
n_neurons_h2: 256
n_neurons_h3: 32
activation: relu
learning_rate: 0.01
optimizer: adam
Score: 0.216463603079319

Trial 09 summary
Hyperparameters:
n_neurons_h1: 96
n_neurons_h2: 32
n_neurons_h3: 32
activation: relu
learning_rate: 0.01
optimizer: adam
Score: 0.2549217715859413

Trial 04 summary
Hyperparameters:
n_neurons_h1: 256
n_neurons_h2: 96
n_neurons_h3: 32
activation: relu
learning_rate: 0.0001
optimizer: rmsprop
Score: 0.3176112025976181

Trial 02 summary
Hyperparameters:
n_neurons_h1: 96
n_neurons_h2: 256
n_neurons_h3: 96
activation: relu
learning_rate: 0.0001
optimizer: adam
Score: 0.32484501600265503

Trial 05 summary
Hyperparameters:
n_neurons_h1: 256
n_neurons_h2: 160
n_neurons_h3: 160
activation: tanh
learning_rate: 0.01
optimizer: rmsprop
Score: 0.3429

# Task 4: Now build AND train the best model

Build the model with the best hyperparameter configuration - look at the code from the inclass notebook section:
```
Keras Tuner Final Step: Build a New Model with the Best Hyperparameters
```

Then train the model for a maximum of 50 epochs:
- Use EarlyStopping
- Use ModelCheckpoint.  IMPORTANT: use filepath='final_tuned_model_assignment.keras'!
- Make sure you call "final_model.load_weights('final_model_init.weights.h5')" **before** fitting....
- Also, use the variable "final_results" to store your result history, so we don't confuse it with the above results.

Make sure you also:
- plot the accuracy and loss for test and train, versus epoch
- print out the confusion matrix

Look at the codeblock(s) you used from the inclass notebook Exercise 5.

In [34]:

# Get the optimal hyperparameters (just redo step from above)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
# your code here
final_model = tuner.hypermodel.build(best_hps)
final_model.save_weights('final_model_init.weights.h5')
print("\n--- Final Model Summary ---")
final_model.summary()
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10),
             keras.callbacks.ModelCheckpoint(filepath='final_tuned_model_assignment.keras', 
                                            monitor='val_loss', 
                                            save_best_only=True)]
final_model.load_weights('final_model_init.weights.h5')
final_results = final_model.fit(train_images_temp, 
                                train_labels_cat_temp,
                                epochs=50,
                                batch_size=128,
                                verbose=1,
                                callbacks=callbacks,
                                validation_data=(val_images, val_labels_cat))


--- Final Model Summary ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 192)            │       150,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 208,682 (815.16 KB)

 Trainable params: 208,682 (815.16 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6056 - loss: 1.1859 - val_accuracy: 0.8643 - val_loss: 0.4647
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9304 - loss: 0.2413 - val_accuracy: 0.9471 - val_loss: 0.2219
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9574 - loss: 0.1408 - val_accuracy: 0.9386 - val_loss: 0.2121
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9704 - loss: 0.1042 - val_accuracy: 0.9343 - val_loss: 0.3152
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9706 - loss: 0.0980 - val_accuracy: 0.9386 - val_loss: 0.2724
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9713 - loss: 0.0938 - val_accuracy: 0.9443 - val_loss: 0.2533
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9831 - loss: 0.0571 - val_accuracy: 0.9500 - val_loss: 0.2460
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9860 - loss: 0.0394 - val_accuracy: 0.9471 - val_loss:

In [35]:
df = pd.DataFrame(final_results.history)
df['epoch'] = df.index + 1
display(df.style)

fig = px.line(df, x='epoch', y=['accuracy','val_accuracy'], title='Accuracy vs Epoch')
fig.show("plotly_mimetype")

fig = px.line(df, x='epoch', y=['loss','val_loss'], title='Loss vs Epoch')
fig.show("plotly_mimetype")

trained_model = keras.models.load_model('final_tuned_model_assignment.keras')

test_loss, test_acc = trained_model.evaluate(test_images, test_labels_cat)
print("Test sample loss: ", test_loss, "; Test sample accuracy: ", test_acc)

predictions = trained_model.predict(test_images)

test_preds = np.argmax(predictions, axis=1)

labels = [0,1,2,3,4,5,6,7,8,9]
results_test = calc_performance_multi(test_labels, test_preds, labels)

print("Average recall test:   ", results_test['report_dict']['macro avg']['recall'])
print()
print("The Confusion matrix for test data:")
print(results_test['confusion_matrix_print_table'])

,accuracy,loss,val_accuracy,val_loss,epoch
0,0.770635,0.710181,0.864286,0.464675,1
1,0.933968,0.228177,0.947143,0.221900,2
2,0.949365,0.162551,0.938571,0.212070,3
3,0.971111,0.100584,0.934286,0.315168,4
4,0.966032,0.107581,0.938571,0.272421,5
5,0.974127,0.085780,0.944286,0.253306,6
6,0.982540,0.059065,0.950000,0.246049,7
7,0.982381,0.052171,0.947143,0.280854,8
8,0.974127,0.091318,0.930000,0.323974,9
9,0.983016,0.057484,0.931429,0.360002,10


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9235 - loss: 0.2560
Test sample loss:  0.23100203275680542 ; Test sample accuracy:  0.9326000213623047
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 911us/step
Average recall test:    0.9323824476917393

The Confusion matrix for test data:
╒════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╕
│        │   Pred:0 │   Pred:1 │   Pred:2 │   Pred:3 │   Pred:4 │   Pred:5 │   Pred:6 │   Pred:7 │   Pred:8 │   Pred:9 │
╞════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ True:0 │      954 │        0 │        0 │        1 │        0 │        9 │        6 │        6 │        3 │        1 │
├────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┤
│ True:1 │        0 │     1104 │        5 │        4 │        1 │        1 │        8 │        5 │        7 │     

# Task 5: Read in Inclass Best model and Compare to above

In [36]:

final_tuned_model_inclass = keras.models.load_model('final_tuned_model_inclass.keras')
# your code here
test_loss_inclass, test_acc_inclass = final_tuned_model_inclass.evaluate(test_images, test_labels_cat)
print("In-class model - Test sample loss: ", test_loss_inclass, "; Test sample accuracy: ", test_acc_inclass)

predictions_inclass = final_tuned_model_inclass.predict(test_images)

test_preds_inclass = np.argmax(predictions_inclass, axis=1)

labels = [0,1,2,3,4,5,6,7,8,9]
results_test_inclass = calc_performance_multi(test_labels, test_preds_inclass, labels)

print("In-class model - Average recall test:   ", results_test_inclass['report_dict']['macro avg']['recall'])
print()
print("In-class model - Confusion matrix for test data:")
print(results_test_inclass['confusion_matrix_print_table'])
print("\n--- COMPARISON ---")
print(f"Assignment Model (3 layers) - Test Accuracy: {test_acc:.4f}")
print(f"In-class Model (1 layer) - Test Accuracy: {test_acc_inclass:.4f}")
print(f"\nAssignment Model - Average Recall: {results_test['report_dict']['macro avg']['recall']:.4f}")
print(f"In-class Model - Average Recall: {results_test_inclass['report_dict']['macro avg']['recall']:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9270 - loss: 0.2668
In-class model - Test sample loss:  0.23209062218666077 ; Test sample accuracy:  0.9354000091552734
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 805us/step
In-class model - Average recall test:    0.9336343049322672

In-class model - Confusion matrix for test data:
╒════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╕
│        │   Pred:0 │   Pred:1 │   Pred:2 │   Pred:3 │   Pred:4 │   Pred:5 │   Pred:6 │   Pred:7 │   Pred:8 │   Pred:9 │
╞════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ True:0 │      963 │        0 │        1 │        2 │        1 │        1 │       11 │        1 │        0 │        0 │
├────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┤
│ True:1 │        0 │     1122 │        3 │        1 │        0 │  

# Task 6: Optimize the number of hidden layers

In the model we tuned above, we fixed the layers at 3.  Can we also optimize the **number** of hidden layers, in addition to all of the other hyperparameters we are tunning?

Yes.   It is just a little bit tricky.  The only code we need to change is in build_model_tuner:
- You need to add a variable hp_n_layers with a min of 1 and max of 3
- You need to use this to control how many layers you add (this is the tricky part)

Once you figure this out, do the search, then build the best model and evaluate just like Tasks 3,4,5 above.

In [37]:
def build_model_tuner(hp):
    """
    Builds a Keras model with hyperparameters for Keras Tuner.
    """
#
# This is the tuning sections -define all of the parameters you want to tune
#
    # Tuning the number of layers
    hp_n_layers = hp.Int('n_layers', min_value=1, max_value=3)
    
    # Tuneing the number of neurons in each of the 3 potential hidden layers
    hp_n_neurons_h1 = hp.Int('n_neurons_h1', min_value=32, max_value=256, step=32)
    hp_n_neurons_h2 = hp.Int('n_neurons_h2', min_value=32, max_value=256, step=32)
    hp_n_neurons_h3 = hp.Int('n_neurons_h3', min_value=32, max_value=256, step=32)
    
    # Tuning the activation function
    hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
        
    # Tuning the learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-1, 1e-2, 1e-3, 1e-4])
    
    # Tuning the optimizer
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd'])
#
# Selecting optimizer based on the choice
    if hp_optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    elif hp_optimizer == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=hp_learning_rate)
  
#
# Defining the model and layers
    input_shape=[28,28]
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=input_shape))
    model.add(keras.layers.Flatten())
#
# Adding hidden layers based on hp_n_layers - this is the tricky part!
    for i in range(hp_n_layers):
        if i == 0:
            model.add(keras.layers.Dense(units=hp_n_neurons_h1, activation=hp_activation))
        elif i == 1:
            model.add(keras.layers.Dense(units=hp_n_neurons_h2, activation=hp_activation))
        elif i == 2:
            model.add(keras.layers.Dense(units=hp_n_neurons_h3, activation=hp_activation))
#
# Output layer
    model.add(keras.layers.Dense(10, activation='softmax'))     
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
#
# Returns model
    return model

In [38]:
import keras_tuner as kt
t0 = time.time()
tuner_v2 = kt.RandomSearch(
    build_model_tuner,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=2,
    directory='keras_tuner_dir',
    project_name='mnist_variable_layers',
    overwrite=True
)

callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)]

print("\nStarting Keras Tuner search with variable layers...")
tuner_v2.search(train_images_temp, train_labels_cat_temp, 
                epochs=10,
                batch_size=128,
                verbose=1,
                callbacks=callbacks,
                validation_data=(val_images, val_labels_cat))

print("Tuner search time:", time.time()-t0)

Trial 10 Complete [00h 00m 06s]
val_loss: 0.24864079803228378

Best val_loss So Far: 0.2344614490866661
Total elapsed time: 00h 01m 04s
Tuner search time: 64.54186463356018


In [39]:
print("\n--- Keras Tuner Search Complete ---")
tuner_v2.results_summary()

best_hps_v2 = tuner_v2.get_best_hyperparameters(num_trials=1)[0]

print("\n--- Best Hyperparameters Found ---")
print(f"""
Number of Layers: {best_hps_v2.get('n_layers')}
Number of Neurons Layer 1: {best_hps_v2.get('n_neurons_h1')}
Number of Neurons Layer 2: {best_hps_v2.get('n_neurons_h2')}
Number of Neurons Layer 3: {best_hps_v2.get('n_neurons_h3')}
Activation Function: {best_hps_v2.get('activation')}
Optimizer: {best_hps_v2.get('optimizer')}
Learning Rate: {best_hps_v2.get('learning_rate')}
""")


--- Keras Tuner Search Complete ---
Results summary
Results in keras_tuner_dir/mnist_variable_layers
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 07 summary
Hyperparameters:
n_layers: 2
n_neurons_h1: 224
n_neurons_h2: 224
n_neurons_h3: 224
activation: relu
learning_rate: 0.01
optimizer: rmsprop
Score: 0.2344614490866661

Trial 09 summary
Hyperparameters:
n_layers: 2
n_neurons_h1: 256
n_neurons_h2: 32
n_neurons_h3: 224
activation: relu
learning_rate: 0.01
optimizer: rmsprop
Score: 0.24864079803228378

Trial 05 summary
Hyperparameters:
n_layers: 2
n_neurons_h1: 192
n_neurons_h2: 224
n_neurons_h3: 256
activation: relu
learning_rate: 0.1
optimizer: sgd
Score: 0.28648290038108826

Trial 04 summary
Hyperparameters:
n_layers: 2
n_neurons_h1: 256
n_neurons_h2: 160
n_neurons_h3: 192
activation: tanh
learning_rate: 0.0001
optimizer: adam
Score: 0.34359341859817505

Trial 02 summary
Hyperparameters:
n_layers: 2
n_neurons_h1: 128
n_neurons_h2: 224
n_neurons_h3: 192
ac

In [40]:
final_model_v2 = tuner_v2.hypermodel.build(best_hps_v2)
final_model_v2.save_weights('final_model_v2_init.weights.h5')

print("\n--- Final Model Summary ---")
final_model_v2.summary()

callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10),
             keras.callbacks.ModelCheckpoint(filepath='final_tuned_model_variable_layers.keras', 
                                            monitor='val_loss', 
                                            save_best_only=True)]

final_model_v2.load_weights('final_model_v2_init.weights.h5')

final_results_v2 = final_model_v2.fit(train_images_temp, 
                                       train_labels_cat_temp,
                                       epochs=50,
                                       batch_size=128,
                                       verbose=1,
                                       callbacks=callbacks,
                                       validation_data=(val_images, val_labels_cat))


--- Final Model Summary ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 224)            │       175,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 224)            │        50,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         2,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 228,490 (892.54 KB)

 Trainable params: 228,490 (892.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5294 - loss: 2.6227 - val_accuracy: 0.7700 - val_loss: 0.8269
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8794 - loss: 0.3988 - val_accuracy: 0.9214 - val_loss: 0.2846
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9319 - loss: 0.2274 - val_accuracy: 0.9343 - val_loss: 0.2623
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9470 - loss: 0.1633 - val_accuracy: 0.9414 - val_loss: 0.2652
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9640 - loss: 0.1176 - val_accuracy: 0.9029 - val_loss: 0.4390
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9759 - loss: 0.0861 - val_accuracy: 0.9371 - val_loss: 0.2912
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9782 - loss: 0.0681 - val_accuracy: 0.9371 - val_loss: 0.4271
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9821 - loss: 0.0625 - val_accuracy: 0.9486 - val_loss:

In [41]:
df = pd.DataFrame(final_results_v2.history)
df['epoch'] = df.index + 1

fig = px.line(df, x='epoch', y=['accuracy','val_accuracy'], title='Accuracy vs Epoch')
fig.show("plotly_mimetype")

fig = px.line(df, x='epoch', y=['loss','val_loss'], title='Loss vs Epoch')
fig.show("plotly_mimetype")

trained_model_v2 = keras.models.load_model('final_tuned_model_variable_layers.keras')
test_loss_v2, test_acc_v2 = trained_model_v2.evaluate(test_images, test_labels_cat)

predictions_v2 = trained_model_v2.predict(test_images)
test_preds_v2 = np.argmax(predictions_v2, axis=1)

labels = [0,1,2,3,4,5,6,7,8,9]
results_test_v2 = calc_performance_multi(test_labels, test_preds_v2, labels)

print("Test accuracy:", test_acc_v2)
print("Average recall:", results_test_v2['report_dict']['macro avg']['recall'])
print("\nConfusion matrix:")
print(results_test_v2['confusion_matrix_print_table'])

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9141 - loss: 0.2867
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step
Test accuracy: 0.9248999953269958
Average recall: 0.9230176788423481

Confusion matrix:
╒════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╕
│        │   Pred:0 │   Pred:1 │   Pred:2 │   Pred:3 │   Pred:4 │   Pred:5 │   Pred:6 │   Pred:7 │   Pred:8 │   Pred:9 │
╞════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ True:0 │      928 │        0 │        7 │        1 │        5 │        3 │       19 │        6 │        3 │        8 │
├────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┤
│ True:1 │        0 │     1124 │        8 │        0 │        0 │        0 │        2 │        0 │        1 │        0 │
├────────┼──────────┼──────────┼──────────┼──────────┼──────────┼───

In [42]:
print("total notebook time:",time.time()-t00)

total notebook time: 139.187420129776
